# Manufacturing Constraint Learning

This notebook demonstrates two complementary settings:

1. A benchmark setting where the synthetic physical feasibility label is available.
2. An outcome-only setting where constraints are learned from observed yield without using the hidden physical label during training.

The second setting is closer to many industrial applications, where the true operating envelope is not explicitly known.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
SRC = ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from industrial_constraint_learning import ManufacturingConstraintLearner, generate_manufacturing_data

## 1. Generate reproducible synthetic process data

In [ ]:
data = generate_manufacturing_data(n_samples=5000, random_state=42)
data.describe()

In [ ]:
print('Physical feasibility rate:', round(data['physical_feasible'].mean(), 3))
print('High-yield rate:', round((data['yield'] >= 85.0).mean(), 3))

## 2. Benchmark model

The benchmark model is trained on the known synthetic feasibility label. Hyperparameters are selected using stratified cross-validation and average precision, which is useful when feasible observations are relatively rare.

In [ ]:
benchmark = ManufacturingConstraintLearner(data, label_mode='physical_feasibility', random_state=42)
benchmark.tune_hyperparameters(cv_splits=5, scoring='average_precision')
benchmark.best_params_, benchmark.cv_best_score_

In [ ]:
evaluation = benchmark.evaluate()
{
    'balanced_accuracy': evaluation.balanced_accuracy,
    'f1': evaluation.f1,
    'roc_auc': evaluation.roc_auc,
    'average_precision': evaluation.average_precision,
}

In [ ]:
benchmark.plot_boundary_comparison()

In [ ]:
benchmark.plot_roc_pr_curves()

## 3. Outcome-only constraint learning

Here the model receives only temperature, pressure, and a label derived from observed yield. The physical feasibility column is not used to fit the classifier. Because this is synthetic data, we can still compare the learned outcome region with hidden physical truth afterward.

In [ ]:
outcome_only = ManufacturingConstraintLearner(data, label_mode='high_yield', high_yield_threshold=85.0, random_state=42)
outcome_only.tune_hyperparameters(cv_splits=5, scoring='average_precision')
outcome_only.best_params_, outcome_only.cv_best_score_

In [ ]:
yield_eval = outcome_only.evaluate()
truth_eval = outcome_only.evaluate_against_physical_truth()
print('High-yield F1:', round(yield_eval.f1, 3))
print('Hidden-constraint recovery F1:', round(truth_eval.f1, 3))

In [ ]:
outcome_only.plot_boundary_comparison()

## 4. Interpretable operating summaries

Quantile bounds summarize where high-yield observations occurred. They should not be interpreted as exact physical constraints.

In [ ]:
benchmark.learn_high_yield_bounds(quantile_margin=0.01)

In [ ]:
benchmark.best_observed_feasible_point()